In [213]:
# hack: execute this cell to fix tqdm progress bar
from ipywidgets import HBox
import tqdm.notebook

class TqdmHBox(HBox):
    def __repr__(self, pretty=False):
        pbar = getattr(self, 'pbar', None)
        if pbar is None:
            return super(TqdmHBox, self).__repr__()
        return pbar.format_meter(**pbar.format_dict)

tqdm.notebook.TqdmHBox = TqdmHBox

<h1>Evaluate enformer-pytorch on single validation sample deposed on github dataset</h1>

In [1]:
#!pip install enformer-pytorch

In [ ]:
import torch
from enformer_pytorch import Enformer

enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough').cuda()
enformer.eval()

In [8]:
!wget https://github.com/lucidrains/enformer-pytorch/raw/main/data/test-sample.pt

--2023-02-19 08:27:37--  https://github.com/lucidrains/enformer-pytorch/raw/main/data/test-sample.pt
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/lucidrains/enformer-pytorch/main/data/test-sample.pt [following]
--2023-02-19 08:27:37--  https://raw.githubusercontent.com/lucidrains/enformer-pytorch/main/data/test-sample.pt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21139943 (20M) [application/octet-stream]
Saving to: ‘test-sample.pt’

test-sample.pt      100%[===================>]  20.16M  44.2MB/s    in 0.5s    

2023-02-19 08:27:39 (44.2 MB/s) - ‘test-sample.pt’ saved [21139943/21139943]

In [12]:
data = torch.load('test-sample.pt')

In [13]:
data

{'sequence': tensor([[1., 0., 0., 0.],
         [0., 0., 0., 1.],
         [0., 0., 1., 0.],
         ...,
         [0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.]], device='cuda:0'),
 'target': tensor([[0.0992, 0.0927, 0.0183,  ..., 0.0000, 0.0000, 0.0000],
         [0.1113, 0.1686, 0.0340,  ..., 0.0000, 0.9844, 0.0000],
         [0.1432, 0.2322, 0.0185,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0066, 0.0167, 0.0076,  ..., 0.0185, 0.1157, 0.0000],
         [0.0041, 0.0016, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0696, 0.0385, 0.0431,  ..., 0.0000, 0.0000, 0.0000]],
        device='cuda:0')}

In [16]:
data["target"].shape

torch.Size([896, 5313])

In [18]:
seq, target = data['sequence'].cuda(), data['target'].cuda()

In [19]:
with torch.no_grad():
    corr_coef = enformer(
        seq,
        target = target,
        return_corr_coef = True,
        head = 'human'
    )

In [20]:
corr_coef

tensor(0.5963, device='cuda:0')

<h1>Get DNA sequnce for enformer-pytorch sample and compare it with our samples</h1> 

<h2>Find enformer-pytorch sample in hg38 genome</h2>

In [21]:
seq

tensor([[1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        ...,
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        [1., 0., 0., 0.]], device='cuda:0')

In [41]:
import numpy as np
def convert_sequence(encoded_seq):
    BASES_ARR = np.array(['A', 'C', 'G', 'T', 'N'])
    # check that encoding is full of zeros or exactly one-hot
    assert min(encoded_seq.max(axis=0)) == 1.0
    assert np.all(encoded_seq.sum(axis=0) == 1.0)

    encoded_seq = encoded_seq.argmax(axis=0)
    return ''.join(BASES_ARR[encoded_seq])


In [42]:
# min(seq.cpu().numpy().T.max(axis=0))
seq_str = convert_sequence(seq.cpu().numpy().T)

In [45]:
seq_str[-1000:]

'TAAAAAGTGGGCAAAGGACCTGAATAAATATTTCTCAAAAGAAGACATACAAACGGCCAAGAAGTACATGAAAAATGCTCAAGATCACTAATCATTGTGGAAAAGCAAATCAAAATCACAATGAGATATTGTCTCACTCCATCCAGTTAGAATGGCTACTATCGAAAAGACAAAAAATTACAAATGTTGGTGAGGATGCAGAGAAAAGGAAACTCTCATACACTCTTAGTAGGAATGCAAATTAGTACAACCATATAGAAAACAATATGAAAGTTCCACAAAAGACTAAAAATAGAATGACCATACAATCCAGCAATTCCACTACTGGGTATTTATCCGAAGGAAAGGAAATTAGTATATCAAAGGGATACCTGCATCCCTGTGTTTCTTGCAGCACAACTCACAATAGCCAAGATATGGAATCAATATAAGTGCCCAGGAGAGGATAAATAGATAAAGAAAATGTGGTATATATACACAATGGAATATTATTCAGCCATAAAAAAACAAAATCTTGCCATTTGTAGCAACATAGATGGAATGGGAGGTCATTACATTAAGTGAAACAAGCCAGGCACAGAAAGCAAAATATCATGTTCTCACTCATGTGGGAGCTAAAAATGTTGATCTCATGGAGATACACAGTAGAATGATTGTCATCAGAGGCTGGGAAAGGCAGCGGAAATGGGAGATGAAGAGAAGTTGACTAATGGGCACACACATGCAGTTACACAGGAGGAAGAATTCTACTGTTCGATGGTAGAGTGACGATAGTTAATAATAATTTATTGTATATTTCAAACTAGCTAGAAGAGAAGATTTGAAATGATACACAAAGAAATGATAGATGTTTGAGGTGATGGATATTCTAAATATCCTGATTTGGTCATTACACAGCGTACGCATGTATTGAAAAATCACATCTACCCCATAAATATGTACAATTACTATTTATCAAGAAAAGAGGAAAAGGAATGAATTATTGATATAGGATGGATG

In [52]:
# aligned this string in genome browser
# https://genome.ucsc.edu/cgi-bin/hgBlat
# to find genomic coordinates of this sequence in hg38
165871274-165740203 #chr6

131071

In [53]:
len(seq_str)

131072

<h2>Find corresponding sample in our dataset</h2>

In [148]:
import h5py

fname = "/mnt/nfs_dna/DNALM/downstream_tasks/enformer/human/h5/human_valid.h5"

f = h5py.File(fname, "r")
f["0"].attrs["coordinates"]

'chr6:165707435-165904042'

In [149]:
# length of our sample is 196607
165707435-165904042

-196607

In [150]:
# midpoint of the interval from our hdf5 dataset
165707435 + (165904042-165707435) // 2

165805738

In [151]:
# midpoint of the interval from pytorch github example
165740203 + (165871274 - 165740203) // 2

165805738

In [79]:
# midpoint of the interval from bed file
165740202 + (165871274 - 165740202) // 2

165805738

<h2>Compare targets between enformer-pytorch and our dataset sample</h2>

In [74]:
f["0"]["target"][0,:10]

array([0.09924316, 0.0927124 , 0.01834106, 0.11224365, 0.07751465,
       0.06982422, 0.13354492, 0.10339355, 0.03744507, 0.12915039],
      dtype=float32)

In [77]:
data["target"][0,:10]

tensor([0.0992, 0.0927, 0.0183, 0.1122, 0.0775, 0.0698, 0.1335, 0.1034, 0.0374,
        0.1292], device='cuda:0')

In [155]:
np.allclose(f["0"]["target"], data["target"].cpu().numpy(), 0.0001)

True

<h1>Run pretrained github pytorch-enformer model on our sample and compare results</h1>

In [102]:
our_record_seq = f["0"]["seq"][()].decode()
our_record_seq[:10]

'CCTCACCTCA'

In [156]:
# check that pytorch-enformer input sequence is substing of our input sequence
our_record_seq.find(seq_str)

32768

In [127]:
def one_hot_encode(seq):
    """
    Given a DNA sequence, return its one-hot encoding
    """
    # Make sure seq has only allowed bases
    allowed = set("ACTGN")
    if not set(seq).issubset(allowed):
        invalid = set(seq) - allowed
        raise ValueError(f"Sequence contains chars not in allowed DNA alphabet (ACGTN): {invalid}")
        
    # Dictionary returning one-hot encoding for each nucleotide 
    nuc_d = {'A':[1.0,0.0,0.0,0.0],
             'C':[0.0,1.0,0.0,0.0],
             'G':[0.0,0.0,1.0,0.0],
             'T':[0.0,0.0,0.0,1.0],
             'N':[0.0,0.0,0.0,0.0]}
    
    # Create array from nucleotide sequence
    vec=np.array([nuc_d[x] for x in seq])
        
    return vec

In [157]:
# DEBUG
one_hot_encode(["A","A","T"])

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.]])

In [158]:
one_hot_encode(our_record_seq)[:10]

array([[0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.]])

In [137]:
our_record_seq_enc = one_hot_encode(our_record_seq)
our_record_seq_enc = torch.from_numpy(our_record_seq_enc).to(target.device,
                                                            dtype=torch.float32)

In [138]:
our_record_seq_enc.size()

torch.Size([196608, 4])

In [159]:
data["sequence"].size()

torch.Size([131072, 4])

In [161]:
# trained model on our sequence and github's targets
with torch.no_grad():
    corr_coef = enformer(
        our_record_seq_enc,
        target = target,
        return_corr_coef = True,
        head = 'human'
    )
corr_coef

tensor(0.6048, device='cuda:0')

In [162]:
# trained model on our sequence and our targets
with torch.no_grad():
    corr_coef = enformer(
        our_record_seq_enc,
        target = torch.from_numpy(f["0"]["target"][()]).to(target.device,
                                                    dtype=torch.float32),
        return_corr_coef = True,
        head = 'human'
    )
corr_coef

tensor(0.6048, device='cuda:0')

In [163]:
# trained model on original sequence and original targets
with torch.no_grad():
    corr_coef = enformer(
        data["sequence"],
        data["target"],
        return_corr_coef = True,
        head = 'human'
    )
corr_coef

tensor(0.5963, device='cuda:0')

<h1>Obtain and behnchmark enformer predictions</h1>

<h2>Some tests</h2>

In [189]:
import tensorflow.compat.v2 as tf
import tensorflow_hub as hub

enformer_model = hub.load("https://tfhub.dev/deepmind/enformer/1").model

TensorShape([1, 896, 1643])

In [ ]:
# Input array [batch_size, SEQ_LENGTH, 4] one hot encoded in order 'ACGT'. The
# `one_hot_encode`. With N values being all zeros.
# inputs = our_record_seq_extended.unsqueeze(dim=0).cpu()
inputs = np.expand_dims(our_record_seq_extended_encoded, axis=0)
predictions = enformer_model.predict_on_batch(inputs)
predictions['human'].shape  # [batch_size, 896, 5313]
predictions['mouse'].shape  # [batch_size, 896, 1643]

In [175]:
inputs = data["sequence"].unsqueeze(dim=0).cpu()
inputs.shape

torch.Size([1, 131072, 4])

In [176]:
our_record_seq_enc.unsqueeze(dim=0).size()

torch.Size([1, 196608, 4])

In [179]:
393216 - 196608

196608

In [180]:
our_record_seq_extended = "".join(["N"]*(196608//2)) + our_record_seq + "".join(["N"]*(196608//2))

In [182]:
c

In [185]:
our_record_seq_extended_encoded.shape

(393216, 4)

<h2>inference</h2>

In [221]:
import h5py
import os
from scipy.stats import pearsonr
import datetime
now = datetime.datetime.now

fname = "/mnt/nfs_dna/DNALM/downstream_tasks/enformer/human/h5/human_test.h5"
out_file = os.path.splitext(fname)[0] + "_predictsEnformerTFhub.h5"

pearsons = []

with h5py.File(fname, "r") as f, \
     h5py.File(out_file, "w") as out:
   for key,val in tqdm.tqdm(f.items()):
    # for key,val in f.items():
        # get seq
        seq = val["seq"][()].decode()
        seq_extended = "".join(["N"]*(196608//2)) + seq + "".join(["N"]*(196608//2))
        seq_extended_encoded = one_hot_encode(seq_extended)
        inputs = np.expand_dims(seq_extended_encoded, axis=0)

        # compute predictions
        predictions = enformer_model.predict_on_batch(inputs)
        predictions = np.squeeze(predictions['human'])

        # compare with targets
        target = np.array(val["target"][()])
        assert predictions.shape == target.shape
        r = pearsonr(target.flatten(), predictions.flatten())
        pearsons.append(r[0])

        # save predictions to the file
        predictions = tf.cast(predictions, tf.float32)
        g = out.create_group(key)
        g.create_dataset('predictions', shape=predictions.shape, data=predictions)
        g.attrs['coordinates'] = val.attrs['coordinates']
        g.attrs['pearson_r'] = r[0]

100%|█████████████████████████████████████| 1937/1937 [6:13:11<00:00, 11.56s/it]


In [222]:
np.mean(pearsons)

0.6937125756651874

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [228]:
test = h5py.File(out_file, "r")
for key,val in test.items():
    x=np.array(val["predictions"][()])
    assert x.shape == (896, 5313)
    print (val.attrs["coordinates"])
    print (val.attrs["pearson_r"])
    break

chr10:37522770-37719377
0.6121044267833071


In [230]:
fname = "/mnt/nfs_dna/DNALM/downstream_tasks/enformer/human/h5/human_train.h5"
f = h5py.File(fname, "r")
len(f.values())

34012

<h1>Run enformer-pytorch metrics on our dataset</h1>

In [ ]:
from enformer_pytorch.metrics import MeanPearsonCorrCoefPerChannel
import h5py
import os
from scipy.stats import pearsonr
import datetime
now = datetime.datetime.now

fname = "/mnt/nfs_dna/DNALM/downstream_tasks/enformer/human/h5/human_test.h5"

corr_coef = MeanPearsonCorrCoefPerChannel(n_channels=5313)

with h5py.File(fname, "r") as f:
    for key,val in tqdm.tqdm(f.items()):
        seq = val["seq"][()].decode()
        seq_extended = "".join(["N"]*(196608//2)) + seq + "".join(["N"]*(196608//2))
        seq_extended_encoded = one_hot_encode(seq_extended)
        inputs = np.expand_dims(seq_extended_encoded, axis=0)

        # compute predictions
        predictions = enformer_model.predict_on_batch(inputs)["human"]
        target = np.expand_dims(np.array(val["target"][()]), axis=0)
        assert predictions.shape == target.shape
        
        # compare with targets
        corr_coef(preds=torch.from_numpy(predictions.numpy()), 
                  target=torch.from_numpy(target)
        )
    
computed = corr_coef.compute()
computed.mean()

 21%|███████▍                            | 399/1937 [1:18:42<5:03:34, 11.84s/it]